In [ ]:
import kagglehub


path = kagglehub.dataset_download("rakkesharv/real-estate-data-from-7-indian-cities")

print("Path to dataset files:", path)

100%|██████████| 1.59M/1.59M [00:00<00:00, 78.3MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/rakkesharv/real-estate-data-from-7-indian-cities/versions/2


In [ ]:
import os

dataset_path = path
print(f"Contents of the dataset directory '{dataset_path}':")
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')

Contents of the dataset directory '/kaggle/input/real-estate-data-from-7-indian-cities':
real-estate-data-from-7-indian-cities/
    Real Estate Data V21.csv


In [ ]:
import pandas as pd

csv_file_path = os.path.join(dataset_path, 'Real Estate Data V21.csv')

df = pd.read_csv(csv_file_path)

print(df.head())
print(df.info())

                                      Name  \
0                         Casagrand ECR 14   
1    Ramanathan Nagar, Pozhichalur,Chennai   
2                              DAC Prapthi   
3  Naveenilaya,Chepauk, Triplicane,Chennai   
4                 VGN Spring Field Phase 1   

                                      Property Title     Price  \
0  4 BHK Flat for sale in Kanathur Reddikuppam, C...  ₹1.99 Cr   
1  10 BHK Independent House for sale in Pozhichal...  ₹2.25 Cr   
2      3 BHK Flat for sale in West Tambaram, Chennai   ₹1.0 Cr   
3  7 BHK Independent House for sale in Triplicane...  ₹3.33 Cr   
4              2 BHK Flat for sale in Avadi, Chennai   ₹48.0 L   

                                   Location  Total_Area  Price_per_SQFT  \
0             Kanathur Reddikuppam, Chennai        2583          7700.0   
1     Ramanathan Nagar, Pozhichalur,Chennai        7000          3210.0   
2  Kasthuribai Nagar, West Tambaram,Chennai        1320          7580.0   
3   Naveenilaya,Chepauk, T

In [ ]:

with open(csv_file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        print(line.strip())

Name,Property Title,Price,Location,Total_Area,Price_per_SQFT,Description,Baths,Balcony
Casagrand ECR 14,"4 BHK Flat for sale in Kanathur Reddikuppam, Chennai",₹1.99 Cr,"Kanathur Reddikuppam, Chennai",2583,7700.0,"Best 4 BHK Apartment for modern-day lifestyle is now available for sale. No brokerage involved, Posted by Owner. Grab this 4 BHK property for sale in one of Chennai's top location, Kanathur Reddikuppam. It is situated on floor 13. The total number of floors in this Apartment is 14. The property price of this unit is Rs 1.99 Cr. The built-up area is 2583 Square feet. There are 4 bedrooms and 4 bathroom. It is an ideal location for young families with kids, as this property is close to Mount Litera Zee School, OMR, Chennai, Amelio Early Education - Siruseri, and Chennai Mathematical Institute. H...",4,Yes
"Ramanathan Nagar, Pozhichalur,Chennai","10 BHK Independent House for sale in Pozhichalur, Chennai",₹2.25 Cr,"Ramanathan Nagar, Pozhichalur,Chennai",7000,3210.0,"Looking for a 

### Data Preprocessing



In [ ]:
import pandas as pd
import os
import numpy as np
import kagglehub
import re


try:
    _ = path
except NameError:
    path = kagglehub.dataset_download("rakkesharv/real-estate-data-from-7-indian-cities")

csv_file_name = 'Real Estate Data V21.csv'
csv_file_path = None

for root, dirs, files in os.walk(path):
    if csv_file_name in files:
        csv_file_path = os.path.join(root, csv_file_name)
        break

if csv_file_path is None:
    raise FileNotFoundError(f"Could not find '{csv_file_name}' in '{path}' or its subdirectories.")

df = pd.read_csv(csv_file_path)

def clean_price(price_str):
    price_str = str(price_str).replace('₹', '').strip()
    if 'Cr' in price_str:
        return float(price_str.replace('Cr', '')) * 1_00_00_000
    elif 'acs' in price_str.lower():
        return np.nan
    elif 'L' in price_str:
        return float(price_str.replace('L', '')) * 1_00_000
    else:
        try:
            return float(price_str)
        except ValueError:
            return np.nan

df['Price'] = df['Price'].apply(clean_price)

def extract_bhk(title):
    match = re.search(r'(\d+)\s*BHK', title, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return np.nan

def extract_kitchen_presence(description):
    if isinstance(description, str) and 'kitchen' in description.lower():
        return 1
    return 0

df['BHK'] = df['Property Title'].apply(extract_bhk)
df['Has_Kitchen'] = df['Description'].apply(extract_kitchen_presence)

df['Balcony'] = df['Balcony'].map({'Yes': 1, 'No': 0})

df_processed = df.drop(columns=['Name', 'Description', 'Property Title'])

df_processed.dropna(subset=['Price'], inplace=True)

if df_processed['BHK'].isnull().any():
    mode_bhk = df_processed['BHK'].mode()[0]
    df_processed['BHK'].fillna(mode_bhk, inplace=True)
    df_processed['BHK'] = df_processed['BHK'].astype(int)

num_unique_locations = df_processed['Location'].nunique()
print(f"Number of unique locations: {num_unique_locations}")

df_processed = pd.get_dummies(df_processed, columns=['Location'], drop_first=True)

print(df_processed.head())
print(df_processed.info())

/tmp/ipykernel_8370/2887484623.py:64: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_processed['BHK'].fillna(mode_bhk, inplace=True)


Number of unique locations: 7050
        Price  Total_Area  Price_per_SQFT  Baths  Balcony  BHK  Has_Kitchen  \
0  19900000.0        2583          7700.0      4        1    4            0   
1  22500000.0        7000          3210.0      6        1   10            0   
2  10000000.0        1320          7580.0      3        0    3            0   
3  33300000.0        4250          7840.0      5        1    7            0   
4   4800000.0         960          5000.0      3        1    2            0   

   Location_   Manganahalli    Sriram Layout ,Ullal Uppanagar, Bangalore  \
0                                              False                       
1                                              False                       
2                                              False                       
3                                              False                       
4                                              False                       

   Location_   sona Building,Bhayan

### Data Splitting


In [ ]:
from sklearn.model_selection import train_test_split

X = df_processed.drop('Price', axis=1)
y = df_processed['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (11620, 7055)
X_test shape: (2906, 7055)
y_train shape: (11620,)
y_test shape: (2906,)


### Model Training



In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lin_reg_model = LinearRegression()
lin_reg_model.fit(X_train, y_train)

y_pred_lin_reg = lin_reg_model.predict(X_test)


mae_lin_reg = mean_absolute_error(y_test, y_pred_lin_reg)
mse_lin_reg = mean_squared_error(y_test, y_pred_lin_reg)
rmse_lin_reg = np.sqrt(mse_lin_reg)
r2_lin_reg = r2_score(y_test, y_pred_lin_reg)

print("Linear Regression Model Evaluation:")
print(f"  Mean Absolute Error (MAE): {mae_lin_reg:,.2f}")
print(f"  Mean Squared Error (MSE): {mse_lin_reg:,.2f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_lin_reg:,.2f}")
print(f"  R-squared (R2): {r2_lin_reg:.4f}")

In [ ]:

rf_reg_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg_model.fit(X_train, y_train)

y_pred_rf_reg = rf_reg_model.predict(X_test)

mae_rf_reg = mean_absolute_error(y_test, y_pred_rf_reg)
mse_rf_reg = mean_squared_error(y_test, y_pred_rf_reg)
rmse_rf_reg = np.sqrt(mse_rf_reg)
r2_rf_reg = r2_score(y_test, y_pred_rf_reg)

print("\nRandom Forest Regressor Model Evaluation:")
print(f"  Mean Absolute Error (MAE): {mae_rf_reg:,.2f}")
print(f"  Mean Squared Error (MSE): {mse_rf_reg:,.2f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_rf_reg:,.2f}")
print(f"  R-squared (R2): {r2_rf_reg:.4f}")


Random Forest Regressor Model Evaluation:
  Mean Absolute Error (MAE): 461,540.55
  Mean Squared Error (MSE): 38,802,583,019,136.78
  Root Mean Squared Error (RMSE): 6,229,171.94
  R-squared (R2): 0.8639


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_reg_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_reg_model.fit(X_train, y_train)


y_pred_gb_reg = gb_reg_model.predict(X_test)


mae_gb_reg = mean_absolute_error(y_test, y_pred_gb_reg)
mse_gb_reg = mean_squared_error(y_test, y_pred_gb_reg)
rmse_gb_reg = np.sqrt(mse_gb_reg)
r2_gb_reg = r2_score(y_test, y_pred_gb_reg)

print("\nGradient Boosting Regressor Model Evaluation:")
print(f"  Mean Absolute Error (MAE): {mae_gb_reg:,.2f}")
print(f"  Mean Squared Error (MSE): {mse_gb_reg:,.2f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_gb_reg:,.2f}")
print(f"  R-squared (R2): {r2_gb_reg:.4f}")


Gradient Boosting Regressor Model Evaluation:
  Mean Absolute Error (MAE): 1,478,589.22
  Mean Squared Error (MSE): 86,723,994,218,110.50
  Root Mean Squared Error (RMSE): 9,312,571.84
  R-squared (R2): 0.6958


### Ensemble Model (Voting Regressor)


In [ ]:
from sklearn.ensemble import VotingRegressor

estimators = [
    ('lr', lin_reg_model),
    ('rf', rf_reg_model),
    ('gb', gb_reg_model)
]


ensemble_model = VotingRegressor(estimators=estimators, n_jobs=-1)
ensemble_model.fit(X_train, y_train)

y_pred_ensemble = ensemble_model.predict(X_test)

mae_ensemble = mean_absolute_error(y_test, y_pred_ensemble)
mse_ensemble = mean_squared_error(y_test, y_pred_ensemble)
rmse_ensemble = np.sqrt(mse_ensemble)
r2_ensemble = r2_score(y_test, y_pred_ensemble)

print("\nEnsemble Model (Voting Regressor) Evaluation:")
print(f"  Mean Absolute Error (MAE): {mae_ensemble:,.2f}")
print(f"  Mean Squared Error (MSE): {mse_ensemble:,.2f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_ensemble:,.2f}")
print(f"  R-squared (R2): {r2_ensemble:.4f}")


Ensemble Model (Voting Regressor) Evaluation:
  Mean Absolute Error (MAE): 2,522,673.79
  Mean Squared Error (MSE): 109,921,758,726,484.39
  Root Mean Squared Error (RMSE): 10,484,357.81
  R-squared (R2): 0.6145


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import VotingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


weights_grid_values = [
    (0.1, 0.45, 0.45),
    (0.45, 0.1, 0.45),
    (0.45, 0.45, 0.1),
    (0.33, 0.33, 0.34),
    (0.0, 0.5, 0.5)
]

best_weights = None
best_mse = float('inf')
best_model_found = None

print("Starting manual Grid Search for VotingRegressor weights...")

for weights_tuple in weights_grid_values:
    print(f"Testing weights: {weights_tuple}")


    current_voting_regressor = VotingRegressor(estimators=estimators, weights=weights_tuple, n_jobs=-1)

    try:
        current_voting_regressor.fit(X_train, y_train)


        y_pred_train = current_voting_regressor.predict(X_train)
        current_mse = mean_squared_error(y_train, y_pred_train)

        print(f"  MSE on training data with these weights: {current_mse:,.2f}")

        if current_mse < best_mse:
            best_mse = current_mse
            best_weights = weights_tuple
            best_model_found = current_voting_regressor

    except Exception as e:
        print(f"  Error fitting with weights {weights_tuple}: {e}")


print("\nManual Grid Search Complete.")

if best_weights:
    print(f"Best VotingRegressor Weights: {best_weights}")
    print(f"Best (training) MSE with these weights: {best_mse:,.2f}")

    best_ensemble_model = best_model_found
    y_pred_tuned_ensemble = best_ensemble_model.predict(X_test)

    mae_tuned_ensemble = mean_absolute_error(y_test, y_pred_tuned_ensemble)
    mse_tuned_ensemble = mean_squared_error(y_test, y_pred_tuned_ensemble)
    rmse_tuned_ensemble = np.sqrt(mse_tuned_ensemble)
    r2_tuned_ensemble = r2_score(y_test, y_pred_tuned_ensemble)

    print("\nTuned Ensemble Model (Voting Regressor) Evaluation (on test set):")
    print(f"  Best Weights: {best_weights}")
    print(f"  Mean Absolute Error (MAE): {mae_tuned_ensemble:,.2f}")
    print(f"  Mean Squared Error (MSE): {mse_tuned_ensemble:,.2f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse_tuned_ensemble:,.2f}")
    print(f"  R-squared (R2): {r2_tuned_ensemble:.4f}")
else:
    print("No successful model training found.")

Starting manual Grid Search for VotingRegressor weights...
Testing weights: (0.1, 0.45, 0.45)
  MSE on training data with these weights: 20,164,394,919,423.76
Testing weights: (0.45, 0.1, 0.45)
  MSE on training data with these weights: 42,086,930,569,422.62
Testing weights: (0.45, 0.45, 0.1)
  MSE on training data with these weights: 38,848,945,974,611.55
Testing weights: (0.33, 0.33, 0.34)
  MSE on training data with these weights: 31,385,992,553,382.00
Testing weights: (0.0, 0.5, 0.5)
  MSE on training data with these weights: 17,490,998,689,510.80

Manual Grid Search Complete.
Best VotingRegressor Weights: (0.0, 0.5, 0.5)
Best (training) MSE with these weights: 17,490,998,689,510.80

Tuned Ensemble Model (Voting Regressor) Evaluation (on test set):
  Best Weights: (0.0, 0.5, 0.5)
  Mean Absolute Error (MAE): 913,787.46
  Mean Squared Error (MSE): 55,869,884,957,183.41
  Root Mean Squared Error (RMSE): 7,474,616.04
  R-squared (R2): 0.8040
